In [6]:
#r "../task17/bin/Debug/net10.0/task17.dll"
#r "nuget: SkiaSharp.NativeAssets.Linux.NoDependencies, 2.88.9"
#r "nuget: ScottPlot, 5.0.56"


using System;
using System.Collections.Generic;
using System.IO;
using System.Linq;
using System.Threading;
using ScottPlot;
using task17;
using Microsoft.DotNet.Interactive.Formatting;

Formatter.Register(typeof(ScottPlot.Plot), (p, w) =>
    w.Write(((ScottPlot.Plot)p).GetPngHtml(600, 400)), HtmlFormatter.MimeType);

ScottPlot.Fonts.Default = "DejaVu Sans";

Installed Packages ScottPlot, 5.0.56 SkiaSharp.NativeAssets.Linux.NoDependencies, 2.88.9

In [7]:
var executionOrder = new List<int>();

public class RecordingTestCommand : ICommand, ILongRunningCommand
{
    private readonly List<int> _log;
    private readonly int _id;
    private readonly int _maxCalls;
    private readonly CountdownEvent _completionSignal;
    private int _counter = 0;

    public RecordingTestCommand(List<int> log, int id, int maxCalls, CountdownEvent completionSignal)
    {
        _log = log;
        _id = id;
        _maxCalls = maxCalls;
        _completionSignal = completionSignal;
    }

    public bool Finished => _counter >= _maxCalls;

    public void Execute()
    {
        _counter++;
        lock (_log) { _log.Add(_id); }
        if (Finished)
            _completionSignal.Signal();
    }
}

In [8]:
var executionOrder = new List<int>();

public class RecordingTestCommand : task17.ICommand, task17.ILongRunningCommand
{
    private readonly List<int> _log;
    private readonly int _id;
    private readonly int _maxCalls;
    private readonly CountdownEvent _completionSignal;
    private readonly task17.IScheduler _scheduler;
    private int _counter = 0;

    public RecordingTestCommand(List<int> log, int id, int maxCalls, CountdownEvent completionSignal, task17.IScheduler scheduler)
    {
        _log = log;
        _id = id;
        _maxCalls = maxCalls;
        _completionSignal = completionSignal;
        _scheduler = scheduler;
    }

    public bool Finished => _counter >= _maxCalls;

    public void Execute()
    {
        _counter++;
        lock (_log) { _log.Add(_id); }
        
        if (!Finished)
        {
            _scheduler.Add(this);
        }
        else
        {
            _completionSignal.Signal();
        }
    }
}

var scheduler = new task17.RoundRobinScheduler();
var server = new task17.ServerThread(scheduler);
var completed = new CountdownEvent(5);
for (int id = 1; id <= 5; id++)
{
    server.Enqueue(new RecordingTestCommand(executionOrder, id, maxCalls: 3, completed, scheduler));
}

server.Start();
completed.Wait(); 

server.Enqueue(new task17.HardStopCommand(server));
server.UnderlyingThread.Join();

In [ ]:
string filePath = "results.txt";
using (StreamWriter writer = new StreamWriter(filePath))
{
    writer.WriteLine("Порядок фактического выполнения:");
    for (int i = 0; i < executionOrder.Count; i++)
        writer.WriteLine($"{i + 1} - команда {executionOrder[i]}");
    writer.WriteLine();
    writer.WriteLine($"Всего вызовов: {executionOrder.Count}");
}


In [ ]:
var plt = new ScottPlot.Plot();
var xs = Enumerable.Range(1, executionOrder.Count).Select(i => (double)i).ToArray();
var ys = executionOrder.Select(id => (double)id).ToArray();
var scatter = plt.Add.Scatter(xs, ys);
scatter.MarkerSize = 8;
scatter.LineWidth = 2;
scatter.Color = ScottPlot.Colors.Blue;
plt.XLabel("номер вызова Execute");
plt.YLabel("id");
plt.Title("чередование в Round Robin планировщике");
plt.Grid.IsVisible = true;
plt.SavePng("plot.png", 800, 600);